# Deeply Learned Features as Markers of MSC Senescence

**Thesis Chapter 3 — Deeply Learned Cell Morphology**

This notebook tests a single question:

> **Do deeply learned features add senescence signal beyond manually extracted features?**

We focus on **passage age** (not treatments). Three deep-feature sets are evaluated
incrementally against the manual baseline:

1. **Deep morphology** — O2-VAE features from *binary cell masks*. Validated against
   *manual morphological* features.
2. **Deep morphology + manual intensity** — does adding intensity information help?
3. **Fully deep** — a model trained on the *9 autofluorescence channels* per cell,
   learning morphology *and* intensity end-to-end.

Every feature set passes through one **identical evaluation harness** so the comparison
is fair and thesis-defensible:

- Single-cell **binary** young-vs-old classification (logistic regression),
  **donor-grouped** cross-validation (no leakage across cell lines).
- **Feature-contribution analysis** — nested model ΔAUC + grouped permutation
  importance. This is the rigorous answer to *"how much are the features adding?"*.
- **Ordinal** logistic regression (proportional-odds) for the *continuum* view, with
  an explicit test of the proportional-odds assumption.
- **UMAP** panels per feature set with passage-age coloring + age centroids/trajectory.
- An **area / image-scaling control** experiment (deep morph ± normalized area), since
  network input scaling distorts cell size — a key confound for senescence.
- A **MIL scaffold** (bag = well) left as a clearly-marked stub for later.

**Design philosophy:** classic, simple results first. The harness makes it trivial to
add feature sets or swap classifiers later without touching the evaluation logic.

## 0. Configuration

**⚠ VERIFY THIS BLOCK FIRST.** Everything downstream is driven from here. Adjust the
file paths, the key columns, and the morph/intensity column ranges to match your CSVs.

### Inputs expected
- `MANUAL_CSV` — manual features. First columns are metadata
  (`Well, ROI, Treatment, Passage, Repeat, Dose, cell_line`), then feature columns.
  Morphological and intensity features are split by **column index range** (see below).
- `DEEP_MORPH_CSV` — O2-VAE (binary-mask) embeddings, one row per cell.
- `DEEP_FULL_CSV` — fully-deep (9-channel) embeddings, one row per cell.

### Join key
All tables are joined on a composite cell key:
**cell_line + passage + well + cell-number-within-well**.
The exact column names in each file are declared below and **printed for you to verify**
in the next cell.

In [ ]:
import os
from dataclasses import dataclass, field
from typing import List, Optional

# ──────────────────────────────────────────────────────────────────────────
# FILE PATHS — EDIT THESE
# ──────────────────────────────────────────────────────────────────────────
MANUAL_CSV     = "data/manualcellsfeats.csv"
DEEP_MORPH_CSV = "data/deep_morph_features.csv"   # O2-VAE, binary masks
DEEP_FULL_CSV  = "data/deep_full_features.csv"    # 9-channel autofluorescence model

OUTPUT_DIR     = "outputs"                          # figures + result tables saved here
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────
# JOIN KEY — the four fields that uniquely identify a cell.
# Provide the column name AS IT APPEARS in each file. If a file uses a different
# name for the same concept, list it under the matching *_aliases entry.
# Set to None if a file does not carry that field directly (we then derive it).
# ──────────────────────────────────────────────────────────────────────────
@dataclass
class KeySpec:
    cell_line: str            # donor / cell line  (grouping unit for CV)
    passage:   str            # passage number (integer-like)
    well:      str            # well identifier (e.g. well_name or 'R..-C..')
    cell_num:  str            # cell / ROI number within the well

# Manual features file key columns (from your attached pipeline):
MANUAL_KEY = KeySpec(cell_line="cell_line", passage="Passage", well="Well", cell_num="ROI")

# Deep-feature files: LABEL THESE so you can verify. If your deep CSVs already
# carry these columns, set the names. If they instead carry a single combined
# key column, set DEEP_*_KEY = None and fill DEEP_*_COMBINED_KEY below.
DEEP_MORPH_KEY = KeySpec(cell_line="cell_line", passage="passage", well="well_name", cell_num="cell")
DEEP_FULL_KEY  = KeySpec(cell_line="cell_line", passage="passage", well="well_name", cell_num="cell")

# Optional: if a deep file has ONE pre-joined key string instead of 4 columns,
# set its name here (must match a key we can also build from the manual file).
DEEP_MORPH_COMBINED_KEY: Optional[str] = None
DEEP_FULL_COMBINED_KEY:  Optional[str] = None

# Prefix used for embedding columns in the deep files. Any column whose name is
# NOT a key/metadata column is treated as an embedding feature, but if your deep
# files prefix them (e.g. 'latent_0'), set it here to be safe.
DEEP_EMB_PREFIX = "latent_"   # set to "" to auto-detect non-key numeric columns

# ──────────────────────────────────────────────────────────────────────────
# MANUAL FEATURE BLOCKS — split by COLUMN INDEX RANGE (per your code).
# In the attached notebook metadata = cols[:7], features = cols[7:].
# Here we further split the FEATURE region into morphology vs intensity.
# Indices are 0-based and refer to the FULL manual dataframe columns.
#   metadata : [0, META_END)
#   morph    : [MORPH_START, MORPH_END)
#   intensity: [INTENS_START, INTENS_END)
# EDIT to match your file. Verified against the print-out in the next cell.
# ──────────────────────────────────────────────────────────────────────────
META_END     = 7      # first 7 cols are metadata
MORPH_START  = 7      # morphological features start
MORPH_END    = 22     # ... end (exclusive)  <-- EDIT
INTENS_START = 22     # intensity features start <-- EDIT
INTENS_END   = None   # None = to the end of the dataframe

# Column carrying cell AREA (used in the scaling/area control experiment).
AREA_COL = "Area"

# ──────────────────────────────────────────────────────────────────────────
# LABELS — passage age scaled to [0,1]; young vs old by thresholds.
# Matches the MIL ordinal convention: 0 = youngest, 1 = oldest.
# ──────────────────────────────────────────────────────────────────────────
YOUNG_MAX = 0.30      # scaled age <= 0.30  -> young (class 0)
OLD_MIN   = 0.70      # scaled age >= 0.70  -> old   (class 1)
# cells with 0.30 < age < 0.70 are EXCLUDED from the binary task (middle band).

# ──────────────────────────────────────────────────────────────────────────
# CV + reproducibility
# ──────────────────────────────────────────────────────────────────────────
RANDOM_STATE = 42
N_SPLITS     = 5          # grouped CV folds (groups = cell_line / donor)
DROP_COLLAPSED_VAR = 0.01 # drop latent dims with variance below this

print("Config loaded. Verify the schema in the next cell before running the analysis.")

## 1. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score
from sklearn.inspection import permutation_importance
from sklearn.base import clone

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(RANDOM_STATE)

# UMAP is optional at import time so the analysis harness still runs without it.
try:
    import umap
    HAVE_UMAP = True
except Exception as e:
    HAVE_UMAP = False
    print(f"[warn] umap not available ({e}); UMAP cells will be skipped.")

pd.set_option("display.max_columns", 60)
print("Imports ready. UMAP available:", HAVE_UMAP)

## 2. Load data and verify the schema

This cell loads all three CSVs and **prints the columns it will treat as key,
metadata, morphology, intensity, and embeddings**. Read the print-out and confirm
it is correct before going further. Nothing here depends on a model — the deep
features are read from CSV.

In [ ]:
def _load(path, name):
    if not os.path.exists(path):
        print(f"[MISSING] {name}: {path}  -> create/point this file, then re-run.")
        return None
    df = pd.read_csv(path)
    print(f"[ok] {name}: {df.shape[0]} rows x {df.shape[1]} cols  ({path})")
    return df

manual_raw     = _load(MANUAL_CSV, "manual features")
deep_morph_raw = _load(DEEP_MORPH_CSV, "deep morph (O2-VAE mask)")
deep_full_raw  = _load(DEEP_FULL_CSV, "deep full (9-channel)")

if manual_raw is not None:
    cols = list(manual_raw.columns)
    meta_cols  = cols[:META_END]
    morph_cols = cols[MORPH_START:MORPH_END]
    intens_cols = cols[INTENS_START:INTENS_END] if INTENS_END else cols[INTENS_START:]
    print("\n--- MANUAL FEATURE SCHEMA (verify!) ---")
    print(f"metadata   [{0}:{META_END}]      = {meta_cols}")
    print(f"morphology [{MORPH_START}:{MORPH_END}] = {morph_cols}")
    print(f"intensity  [{INTENS_START}:{INTENS_END}] = {intens_cols}")
    assert AREA_COL in cols, f"AREA_COL '{AREA_COL}' not found in manual features."
    print(f"\nArea column present: '{AREA_COL}' ✔")

## 3. Build a common cell key

Every table gets a single canonical string key:
`cellline | passage | well | cellnum`, normalized (stripped, lower-cased, passage
cast to int). This is the join backbone. If your deep files use a single combined
key column, set the `DEEP_*_COMBINED_KEY` options in the config instead.

In [ ]:
def _norm(s):
    return s.astype(str).str.strip().str.lower()

def _norm_passage(s):
    # robustly pull an integer passage out of values like 'P7', '07', 7.0
    return (s.astype(str)
             .str.extract(r"(\d+)", expand=False)
             .astype(float).astype("Int64").astype(str))

def build_key(df, spec, combined=None, well_normalizer=None):
    """Attach a 'cell_key' column to df."""
    df = df.copy()
    if combined is not None and combined in df.columns:
        df["cell_key"] = _norm(df[combined])
        return df
    well = _norm(df[spec.well])
    if well_normalizer is not None:
        well = well_normalizer(well)
    df["cell_key"] = (
        _norm(df[spec.cell_line]) + "|" +
        _norm_passage(df[spec.passage]) + "|" +
        well + "|" +
        _norm(df[spec.cell_num])
    )
    return df

# OPTIONAL well-name normalizer: deep & manual files may store wells differently
# (e.g. "DeHBSS_..._P7 - R02-C03.tif" vs "R02-C03"). If so, write a function that
# reduces both to the same canonical token (e.g. the 'R..-C..' substring).
import re
def canonical_well(well_series):
    out = well_series.str.extract(r"(r\d+-?c\d+|r\d+\s*-\s*c\d+)", expand=False)
    return out.fillna(well_series)  # fall back to raw if no R/C pattern

WELL_NORMALIZER = None   # set to `canonical_well` if your well strings differ

manual = build_key(manual_raw, MANUAL_KEY, well_normalizer=WELL_NORMALIZER) if manual_raw is not None else None
deep_morph = (build_key(deep_morph_raw, DEEP_MORPH_KEY, DEEP_MORPH_COMBINED_KEY, WELL_NORMALIZER)
              if deep_morph_raw is not None else None)
deep_full = (build_key(deep_full_raw, DEEP_FULL_KEY, DEEP_FULL_COMBINED_KEY, WELL_NORMALIZER)
             if deep_full_raw is not None else None)

for nm, df in [("manual", manual), ("deep_morph", deep_morph), ("deep_full", deep_full)]:
    if df is not None:
        n_dup = df["cell_key"].duplicated().sum()
        print(f"{nm}: {df['cell_key'].nunique()} unique keys"
              f"  (sample: {df['cell_key'].iloc[0]!r}); duplicate keys: {n_dup}")

## 4. Merge into one aligned table

We do an **inner** join of manual features with each deep-feature table on `cell_key`,
keeping a single tidy frame per analysis. Unmatched cells are reported (a high
unmatched count usually means the well/cell-number conventions differ — fix the
`KeySpec` / `WELL_NORMALIZER` above).

In [ ]:
def _emb_cols(df, spec, combined):
    key_like = {spec.cell_line, spec.passage, spec.well, spec.cell_num, "cell_key"}
    if combined: key_like.add(combined)
    if DEEP_EMB_PREFIX:
        emb = [c for c in df.columns if c.startswith(DEEP_EMB_PREFIX)]
        if emb:
            return emb
    # auto-detect: numeric, non-key columns
    return [c for c in df.columns
            if c not in key_like and pd.api.types.is_numeric_dtype(df[c])]

def merge_deep(manual_df, deep_df, spec, combined, tag):
    emb = _emb_cols(deep_df, spec, combined)
    deep_small = deep_df[["cell_key"] + emb].copy()
    deep_small = deep_small.rename(columns={c: f"{tag}_{i}" for i, c in enumerate(emb)})
    merged = manual_df.merge(deep_small, on="cell_key", how="inner")
    matched = len(merged); total_manual = len(manual_df)
    print(f"[{tag}] embeddings: {len(emb)} dims | matched {matched}/{total_manual} manual cells "
          f"({100*matched/max(total_manual,1):.1f}%)")
    return merged, [f"{tag}_{i}" for i in range(len(emb))]

if manual is not None and deep_morph is not None:
    df_morph, deepmorph_cols = merge_deep(manual, deep_morph, DEEP_MORPH_KEY, DEEP_MORPH_COMBINED_KEY, "dmorph")
if manual is not None and deep_full is not None:
    df_full, deepfull_cols = merge_deep(manual, deep_full, DEEP_FULL_KEY, DEEP_FULL_COMBINED_KEY, "dfull")

# A combined frame that has BOTH deep sets aligned to the same cells (intersection),
# used for the head-to-head feature-set comparison.
if manual is not None and deep_morph is not None and deep_full is not None:
    df_all = df_morph.merge(df_full[["cell_key"] + deepfull_cols], on="cell_key", how="inner")
    print(f"\n[all] cells with BOTH deep sets + manual: {len(df_all)}")

## 5. Labels: ordinal age + young/old binary

Passage is scaled to **[0, 1]** (0 = youngest passage present, 1 = oldest), matching
the MIL convention. From the scaled age:

- **young (class 0)**: scaled age ≤ `YOUNG_MAX` (0.30)
- **old (class 1)**: scaled age ≥ `OLD_MIN` (0.70)
- the middle band is **excluded** from the binary task (it is kept for ordinal/UMAP).

The donor/cell-line column is the **grouping variable** for cross-validation, so no
cell line appears in both train and test — the key guard against optimistic leakage.

In [ ]:
def add_labels(df):
    df = df.copy()
    p = pd.to_numeric(df[MANUAL_KEY.passage].astype(str).str.extract(r"(\d+)", expand=False),
                      errors="coerce")
    pmin, pmax = p.min(), p.max()
    df["passage_int"] = p
    df["age_scaled"] = (p - pmin) / (pmax - pmin) if pmax > pmin else 0.0
    df["group"] = _norm(df[MANUAL_KEY.cell_line])  # donor / cell line for grouped CV

    # binary label with middle band excluded
    df["y_bin"] = np.nan
    df.loc[df["age_scaled"] <= YOUNG_MAX, "y_bin"] = 0
    df.loc[df["age_scaled"] >= OLD_MIN,  "y_bin"] = 1

    # ordinal label = ranked passage index (0..K-1), for ordinal regression
    order = {v: i for i, v in enumerate(sorted(df["passage_int"].dropna().unique()))}
    df["y_ord"] = df["passage_int"].map(order)
    return df, pmin, pmax

for nm in ["df_morph", "df_full", "df_all"]:
    if nm in dir():
        labeled, pmin, pmax = add_labels(eval(nm))
        globals()[nm] = labeled

if "df_all" in dir():
    d = df_all
    print(f"Passage range: P{int(pmin)}..P{int(pmax)}  (scaled 0..1)")
    print("\nPassage distribution:")
    print(d.groupby("passage_int").size())
    print("\nBinary label counts (young=0, old=1; NaN=middle excluded):")
    print(d["y_bin"].value_counts(dropna=False))
    print("\nCells per donor/group:")
    print(d.groupby("group").size())

## 6. Feature-set registry

This is the heart of the comparison. Each entry names the columns that make up a
feature set. The harness below evaluates them all the *same way*, so adding a new
set later is a one-line change.

The incremental story (per the thesis narrative):

| key | what it is | role |
|---|---|---|
| `manual_morph` | manual morphological block | baseline shape |
| `manual_intensity` | manual intensity block | baseline intensity |
| `manual_all` | morph + intensity | full manual baseline |
| `deep_morph` | O2-VAE mask embeddings | **validate vs manual morph** |
| `deep_morph+manual_intensity` | learned shape + manual intensity | does intensity help? |
| `deep_full` | 9-channel learned (morph+intensity) | end-to-end deep |

`deep_morph_area` adds normalized `Area` to the deep-morph block — the scaling/size
control experiment.

In [ ]:
# Resolve the manual block column lists from the index ranges (config section 0).
# IMPORTANT: derive from the ORIGINAL raw manual columns, BEFORE any key/label
# columns were appended, so an open-ended intensity slice can't sweep them up.
_cols = list(manual_raw.columns)
MORPH_COLS  = _cols[MORPH_START:MORPH_END]
INTENS_COLS = _cols[INTENS_START:INTENS_END] if INTENS_END else _cols[INTENS_START:]
MANUAL_ALL  = MORPH_COLS + INTENS_COLS

# Safety: never let bookkeeping columns leak into a feature block.
_FORBIDDEN = {"cell_key", "group", "y_bin", "y_ord", "age_scaled", "passage_int",
              MANUAL_KEY.cell_line, MANUAL_KEY.passage, MANUAL_KEY.well, MANUAL_KEY.cell_num}
MORPH_COLS  = [c for c in MORPH_COLS  if c not in _FORBIDDEN]
INTENS_COLS = [c for c in INTENS_COLS if c not in _FORBIDDEN]
MANUAL_ALL  = [c for c in MANUAL_ALL  if c not in _FORBIDDEN]
print(f"Resolved blocks -> morph: {len(MORPH_COLS)}, intensity: {len(INTENS_COLS)}")

def make_registry(df, has_morph=True, has_full=True):
    """Build {name: [cols]} using only columns present in df."""
    reg = {}
    def keep(cols): return [c for c in cols if c in df.columns]
    reg["manual_morph"]      = keep(MORPH_COLS)
    reg["manual_intensity"]  = keep(INTENS_COLS)
    reg["manual_all"]        = keep(MANUAL_ALL)
    if has_morph:
        dm = [c for c in df.columns if c.startswith("dmorph_")]
        reg["deep_morph"] = dm
        reg["deep_morph+manual_intensity"] = dm + keep(INTENS_COLS)
        reg["deep_morph_area"] = dm + ([AREA_COL] if AREA_COL in df.columns else [])
    if has_full:
        dfu = [c for c in df.columns if c.startswith("dfull_")]
        reg["deep_full"] = dfu
    # prune empties
    return {k: v for k, v in reg.items() if len(v) > 0}

REGISTRY = make_registry(df_all)
print("Feature sets available on df_all:")
for k, v in REGISTRY.items():
    print(f"  {k:32s} {len(v):4d} cols")

## 7. Normalization policy

Two safeguards, both folded *inside* the CV pipeline so scalers are fit on training
folds only (no leakage):

1. **Drop collapsed latent dims** (variance < `DROP_COLLAPSED_VAR`). VAE KL pushes
   unused posteriors to N(0,1); collapsed dims add noise to distance metrics. (Mirrors
   the posterior-collapse check in your original notebook.)
2. **Per-feature z-scoring** (`StandardScaler`). Manual features are in arbitrary
   physical units; latent dims are only approximately standardized. Z-scoring both
   blocks prevents either from dominating purely by scale.

A note on **block-size imbalance**: after z-scoring, total variance ≈ number of
features in a block, so a much larger block can dominate distances (relevant for UMAP
and any unscaled distance). We report block sizes and, for UMAP, offer optional
per-block re-weighting.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class DropCollapsed(BaseEstimator, TransformerMixin):
    """Drop near-constant (collapsed) columns, fit on training data only."""
    def __init__(self, thresh=DROP_COLLAPSED_VAR):
        self.thresh = thresh
    def fit(self, X, y=None):
        v = np.asarray(X).var(axis=0)
        self.keep_ = v >= self.thresh
        if self.keep_.sum() == 0:      # safety: keep all if everything looks collapsed
            self.keep_ = np.ones_like(self.keep_, dtype=bool)
        return self
    def transform(self, X):
        return np.asarray(X)[:, self.keep_]

def make_clf():
    """Standard classifier pipeline used for EVERY feature set."""
    return Pipeline([
        ("drop", DropCollapsed()),
        ("scale", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000, class_weight="balanced",
                                  random_state=RANDOM_STATE)),
    ])

## 8. Evaluation harness — donor-grouped binary CV

`StratifiedGroupKFold` keeps the young/old ratio balanced across folds **while
guaranteeing no donor/cell-line appears in both train and test**. For each feature
set we report ROC-AUC, PR-AUC (average precision), and balanced accuracy as
mean ± std across folds. PR-AUC is included because the classes are imbalanced.

In [ ]:
def eval_binary(df, cols, n_splits=N_SPLITS, return_oof=False):
    """Grouped, stratified CV for one feature set. Returns metric dict (+ optional OOF)."""
    sub = df.dropna(subset=["y_bin"]).copy()
    X = sub[cols].astype(float).values
    y = sub["y_bin"].astype(int).values
    g = sub["group"].values

    n_groups = len(np.unique(g))
    splits = min(n_splits, n_groups)
    sgkf = StratifiedGroupKFold(n_splits=splits, shuffle=True, random_state=RANDOM_STATE)

    aucs, aps, baccs = [], [], []
    oof = np.full(len(sub), np.nan)
    for tr, te in sgkf.split(X, y, groups=g):
        clf = make_clf().fit(X[tr], y[tr])
        p = clf.predict_proba(X[te])[:, 1]
        oof[te] = p
        # guard against single-class test folds
        if len(np.unique(y[te])) > 1:
            aucs.append(roc_auc_score(y[te], p))
            aps.append(average_precision_score(y[te], p))
        baccs.append(balanced_accuracy_score(y[te], (p >= 0.5).astype(int)))

    res = dict(
        n_cells=len(sub), n_dims=len(cols), n_groups=n_groups, n_splits=splits,
        roc_auc=np.mean(aucs), roc_auc_std=np.std(aucs),
        pr_auc=np.mean(aps),   pr_auc_std=np.std(aps),
        bal_acc=np.mean(baccs), bal_acc_std=np.std(baccs),
    )
    if return_oof:
        return res, sub, oof
    return res

def run_all(df, names):
    rows = []
    for name in names:
        if name not in REGISTRY:
            print(f"[skip] {name} not in registry"); continue
        r = eval_binary(df, REGISTRY[name]); r["feature_set"] = name
        rows.append(r)
    out = pd.DataFrame(rows).set_index("feature_set")
    return out[["n_cells", "n_dims", "n_groups",
                "roc_auc", "roc_auc_std", "pr_auc", "pr_auc_std",
                "bal_acc", "bal_acc_std"]]

### 8.1 Headline result table

The incremental comparison in one table. Read it top-to-bottom as the thesis story:
manual baselines → deep morph (vs manual morph) → deep morph + intensity → fully deep.

In [ ]:
ORDER = ["manual_morph", "manual_intensity", "manual_all",
         "deep_morph", "deep_morph+manual_intensity", "deep_full"]
results_bin = run_all(df_all, [n for n in ORDER if n in REGISTRY])
results_bin.round(3)

In [ ]:
# Persist + simple bar chart of ROC-AUC with error bars
results_bin.to_csv(os.path.join(OUTPUT_DIR, "binary_results.csv"))

fig, ax = plt.subplots(figsize=(9, 4.5))
r = results_bin
ax.bar(range(len(r)), r["roc_auc"], yerr=r["roc_auc_std"], capsize=4,
       color="#4C72B0", alpha=0.9)
ax.axhline(0.5, ls="--", c="grey", lw=1, label="chance")
ax.set_xticks(range(len(r))); ax.set_xticklabels(r.index, rotation=30, ha="right")
ax.set_ylabel("ROC-AUC (donor-grouped CV)"); ax.set_ylim(0.4, 1.0)
ax.set_title("Young vs Old — single-cell classification by feature set")
ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "binary_roc_auc.png"), dpi=150)
plt.show()

## 9. How much do the deep features actually add?

A higher AUC alone is not enough for a thesis claim. We quantify *incremental*
contribution two ways, both using the **same donor-grouped folds**:

**(a) Nested model ΔAUC.** Compare a baseline feature set to baseline + added block.
The difference in mean CV AUC (ΔAUC), with a paired across-fold spread, tells you the
marginal value of the added block. Key contrasts:

- `manual_morph` → `deep_morph` (does learned shape beat hand-crafted shape?)
- `manual_all` → `deep_morph+manual_intensity` (does learned shape + manual intensity beat full manual?)
- `manual_all` → `deep_full` (does end-to-end deep beat full manual?)
- `deep_morph` → `deep_morph_area` (how much is just *size*? — the scaling control)

**(b) Grouped permutation importance.** Within the best model, permute whole feature
*blocks* (all manual-morph dims together, all deep dims together, area alone) and
measure the AUC drop. Permuting a *block* answers "how much does this *kind* of
feature matter", which is what the thesis argues about — not individual noisy dims.

In [ ]:
def paired_delta_auc(df, base_name, plus_name, n_splits=N_SPLITS):
    """Per-fold AUC for two feature sets on identical splits -> paired ΔAUC."""
    sub = df.dropna(subset=["y_bin"]).copy()
    y = sub["y_bin"].astype(int).values
    g = sub["group"].values
    Xb = sub[REGISTRY[base_name]].astype(float).values
    Xp = sub[REGISTRY[plus_name]].astype(float).values

    splits = min(n_splits, len(np.unique(g)))
    sgkf = StratifiedGroupKFold(n_splits=splits, shuffle=True, random_state=RANDOM_STATE)
    d = []
    for tr, te in sgkf.split(Xb, y, groups=g):
        if len(np.unique(y[te])) < 2:
            continue
        ab = roc_auc_score(y[te], make_clf().fit(Xb[tr], y[tr]).predict_proba(Xb[te])[:, 1])
        ap = roc_auc_score(y[te], make_clf().fit(Xp[tr], y[tr]).predict_proba(Xp[te])[:, 1])
        d.append(ap - ab)
    d = np.array(d)
    return dict(base=base_name, plus=plus_name,
                delta_auc=d.mean(), delta_std=d.std(),
                n_folds=len(d), wins=int((d > 0).sum()))

CONTRASTS = [
    ("manual_morph", "deep_morph"),
    ("manual_all",   "deep_morph+manual_intensity"),
    ("manual_all",   "deep_full"),
    ("deep_morph",   "deep_morph_area"),
]
delta_rows = [paired_delta_auc(df_all, b, p) for b, p in CONTRASTS
              if b in REGISTRY and p in REGISTRY]
delta_df = pd.DataFrame(delta_rows)
delta_df.to_csv(os.path.join(OUTPUT_DIR, "delta_auc_contrasts.csv"), index=False)
delta_df.round(4)

In [ ]:
# (b) Block-level permutation importance on the strongest single set (by ROC-AUC)
best_name = results_bin["roc_auc"].idxmax()
print(f"Block permutation importance for best set: {best_name}")

def block_perm_importance(df, feat_name, n_repeats=20):
    sub = df.dropna(subset=["y_bin"]).copy()
    cols = REGISTRY[feat_name]
    X = sub[cols].astype(float).values
    y = sub["y_bin"].astype(int).values
    g = sub["group"].values

    # define blocks by column-name prefix within this feature set
    def block_of(c):
        if c.startswith("dmorph_"): return "deep_morph"
        if c.startswith("dfull_"):  return "deep_full"
        if c == AREA_COL:           return "area"
        if c in MORPH_COLS:         return "manual_morph"
        if c in INTENS_COLS:        return "manual_intensity"
        return "other"
    blocks = {}
    for i, c in enumerate(cols):
        blocks.setdefault(block_of(c), []).append(i)

    splits = min(N_SPLITS, len(np.unique(g)))
    sgkf = StratifiedGroupKFold(n_splits=splits, shuffle=True, random_state=RANDOM_STATE)
    rng = np.random.default_rng(RANDOM_STATE)
    import collections
    drops = collections.defaultdict(list)
    for tr, te in sgkf.split(X, y, groups=g):
        if len(np.unique(y[te])) < 2:
            continue
        clf = make_clf().fit(X[tr], y[tr])
        base = roc_auc_score(y[te], clf.predict_proba(X[te])[:, 1])
        for bname, idxs in blocks.items():
            accum = []
            for _ in range(n_repeats):
                Xp = X[te].copy()
                perm = rng.permutation(len(te))
                Xp[:, idxs] = Xp[perm][:, idxs]
                accum.append(base - roc_auc_score(y[te], clf.predict_proba(Xp)[:, 1]))
            drops[bname].append(np.mean(accum))
    return pd.DataFrame({b: pd.Series(v) for b, v in drops.items()}).agg(["mean", "std"]).T

perm_df = block_perm_importance(df_all, best_name)
perm_df.columns = ["importance_mean", "importance_std"]
perm_df = perm_df.sort_values("importance_mean", ascending=False)
perm_df.to_csv(os.path.join(OUTPUT_DIR, "block_permutation_importance.csv"))
perm_df.round(4)

## 10. The continuum view — ordinal regression + proportional-odds test

The young/old binary split discards the middle band and imposes a categorical cut on
what is biologically a continuum. Here we model **passage rank** directly with ordinal
logistic regression (proportional-odds / cumulative-logit), and **explicitly test the
proportional-odds (PO) assumption** — which you flagged as important given biological
heterogeneity, class imbalance, and uneven group sizes.

We report ordinal performance via leave-one-donor-out spearman correlation between
predicted expected rank and true rank, plus a likelihood-ratio Brant-style check of PO.
This needs `mord` (ordinal regression) — install with `pip install mord` if missing.

In [ ]:
try:
    import mord
    HAVE_MORD = True
except Exception as e:
    HAVE_MORD = False
    print(f"[warn] mord not installed ({e}). Run `pip install mord` for ordinal models.")

from scipy.stats import spearmanr
from sklearn.model_selection import LeaveOneGroupOut

def eval_ordinal(df, feat_name):
    if not HAVE_MORD:
        return None
    sub = df.dropna(subset=["y_ord"]).copy()
    cols = REGISTRY[feat_name]
    X = sub[cols].astype(float).values
    y = sub["y_ord"].astype(int).values
    g = sub["group"].values

    logo = LeaveOneGroupOut()
    preds = np.full(len(sub), np.nan)
    for tr, te in logo.split(X, y, groups=g):
        pipe = Pipeline([("drop", DropCollapsed()), ("scale", StandardScaler())]).fit(X[tr])
        Xtr, Xte = pipe.transform(X[tr]), pipe.transform(X[te])
        m = mord.LogisticAT(alpha=1.0).fit(Xtr, y[tr])
        # expected rank = sum_k k * P(y=k)
        classes = m.classes_
        proba = m.predict_proba(Xte)
        preds[te] = proba @ classes
    rho, p = spearmanr(preds[~np.isnan(preds)], y[~np.isnan(preds)])
    return dict(feature_set=feat_name, spearman=rho, spearman_p=p, n=int((~np.isnan(preds)).sum()))

if HAVE_MORD:
    ord_rows = [eval_ordinal(df_all, n) for n in
                ["manual_all", "deep_morph", "deep_morph+manual_intensity", "deep_full"]
                if n in REGISTRY]
    ord_df = pd.DataFrame([r for r in ord_rows if r])
    ord_df.to_csv(os.path.join(OUTPUT_DIR, "ordinal_results.csv"), index=False)
    display(ord_df.round(4))

In [ ]:
# Proportional-odds (PO) assumption check via separate binary fits at each cut.
# If PO holds, coefficient signs/magnitudes are roughly stable across cut points.
def po_assumption_check(df, feat_name, max_dims=12):
    sub = df.dropna(subset=["y_ord"]).copy()
    cols = REGISTRY[feat_name][:max_dims]      # limit dims for a readable diagnostic
    Xall = StandardScaler().fit_transform(sub[cols].astype(float).values)
    yo = sub["y_ord"].astype(int).values
    cuts = sorted(np.unique(yo))[:-1]
    coefs = {}
    for c in cuts:
        ybin = (yo > c).astype(int)
        if len(np.unique(ybin)) < 2:
            continue
        lr = LogisticRegression(max_iter=2000).fit(Xall, ybin)
        coefs[f">cut{c}"] = lr.coef_[0]
    coef_df = pd.DataFrame(coefs, index=cols)
    # stability = std of each coefficient across cut points (small = PO plausible)
    coef_df["coef_std_across_cuts"] = coef_df.std(axis=1)
    return coef_df

po_df = po_assumption_check(df_all, "manual_all")
print("PO check (coefficients across cut points; large std => PO may be violated):")
po_df.round(3)

## 11. Image-scaling / area control experiment

A core confound: the network's fixed input size means each cell is rescaled, so the
embeddings may *lose absolute size* — yet cell enlargement is a hallmark of senescence.
This experiment isolates how much of the signal is **size** vs **shape**:

- `deep_morph` (shape only, size partly normalized away by rescaling)
- `deep_morph_area` (shape + explicit normalized Area appended)
- `manual_morph` includes Area natively (reference)

The ΔAUC `deep_morph → deep_morph_area` (computed in §9) is the headline number here;
below we visualize it and report how Area alone classifies young/old.

In [ ]:
# Area-only classifier as a sanity floor
area_only = eval_binary(df_all.assign(**{}), [AREA_COL]) if AREA_COL in df_all.columns else None
if area_only:
    print(f"Area-only ROC-AUC: {area_only['roc_auc']:.3f} ± {area_only['roc_auc_std']:.3f}")

# bar of the area contrast from delta_df
row = delta_df[(delta_df.base=="deep_morph") & (delta_df.plus=="deep_morph_area")]
if len(row):
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(["+Area"], row["delta_auc"].values, yerr=row["delta_std"].values, capsize=5, color="#C44E52")
    ax.axhline(0, c="grey", lw=1)
    ax.set_ylabel("ΔROC-AUC vs deep_morph")
    ax.set_title("Adding normalized Area to deep-morph features")
    fig.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "area_control.png"), dpi=150)
    plt.show()

## 12. UMAP visualization per feature set

For the thesis figure: one UMAP per feature set, colored by passage age, with **age
centroids** and a **centroid trajectory** overlaid. The trajectory makes the
"continuum vs discrete states" argument visual — a smooth path supports a continuum;
well-separated clusters support discrete age states.

Normalization for UMAP mirrors the classifier: drop collapsed dims, z-score per
feature. We fit UMAP on the **whole** dataset for visualization (this is descriptive,
not a held-out predictive claim).

In [ ]:
def umap_embed(df, cols, n_neighbors=50, min_dist=0.4, seed=RANDOM_STATE):
    X = df[cols].astype(float).values
    X = DropCollapsed().fit_transform(X)
    X = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_components=2, n_neighbors=n_neighbors, min_dist=min_dist,
                        metric="euclidean", random_state=seed)
    return reducer.fit_transform(X)

def centroids(emb, ages):
    dfc = pd.DataFrame({"u1": emb[:, 0], "u2": emb[:, 1], "age": ages})
    return dfc.groupby("age")[["u1", "u2"]].mean().sort_index()

def plot_umap_panel(ax, emb, ages, title):
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=ages, cmap="viridis", s=6, alpha=0.6)
    cen = centroids(emb, ages)
    ax.plot(cen["u1"], cen["u2"], "-o", color="crimson", lw=2, ms=6, label="age centroid path")
    for age, r in cen.iterrows():
        ax.annotate(f"P{int(age)}", (r["u1"], r["u2"]), color="crimson",
                    fontsize=9, fontweight="bold")
    ax.set_title(title); ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
    return sc

In [ ]:
if HAVE_UMAP:
    panel_sets = [n for n in ["manual_all", "deep_morph", "deep_morph_area", "deep_full"]
                  if n in REGISTRY]
    ncol = len(panel_sets)
    fig, axes = plt.subplots(1, ncol, figsize=(6*ncol, 5.5), squeeze=False)
    ages = df_all["passage_int"].values
    sc = None
    for ax, name in zip(axes[0], panel_sets):
        emb = umap_embed(df_all, REGISTRY[name])
        sc = plot_umap_panel(ax, emb, ages, name)
    axes[0][0].legend(loc="best", fontsize=8)
    fig.colorbar(sc, ax=axes[0], fraction=0.025, pad=0.01, label="Passage")
    fig.suptitle("UMAP by feature set — passage age + centroid trajectory", y=1.02)
    fig.savefig(os.path.join(OUTPUT_DIR, "umap_panels.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("UMAP not available; install umap-learn to render these panels.")

## 13. MIL scaffold (stub — to fill later)

Single-cell classification is the core result. **Multiple-Instance Learning** is the
scale-up story for the thesis: each *well* is a bag of cells with a single passage-age
label, which matches how senescence manifests as a *proportion* of senescent cells
rather than a per-cell certainty.

This section is intentionally a scaffold — it builds the bags and a baseline pooling
classifier so the wiring is correct, leaving attention-MIL for a later pass so we
don't over-commit now.

In [ ]:
# ---- Bag construction: group cells by well, label = bag passage age ------------
def build_bags(df, cols, bag_key="cell_key_well"):
    sub = df.dropna(subset=["y_bin"]).copy()
    # bag id = donor + passage + well (a single well of one donor at one passage)
    sub["bag_id"] = (sub["group"].astype(str) + "|" +
                     sub["passage_int"].astype(str) + "|" +
                     sub[MANUAL_KEY.well].astype(str))
    bags, bag_y, bag_grp = [], [], []
    for bid, gdf in sub.groupby("bag_id"):
        bags.append(gdf[cols].astype(float).values)   # (n_cells_in_bag, n_dims)
        bag_y.append(int(gdf["y_bin"].iloc[0]))
        bag_grp.append(gdf["group"].iloc[0])
    return bags, np.array(bag_y), np.array(bag_grp)

# Baseline MIL = mean-pool each bag, then the SAME donor-grouped LR harness.
def eval_mil_meanpool(df, feat_name):
    bags, by, bgrp = build_bags(df, REGISTRY[feat_name])
    Xbag = np.vstack([b.mean(axis=0) for b in bags])
    splits = min(N_SPLITS, len(np.unique(bgrp)))
    sgkf = StratifiedGroupKFold(n_splits=splits, shuffle=True, random_state=RANDOM_STATE)
    aucs = []
    for tr, te in sgkf.split(Xbag, by, groups=bgrp):
        if len(np.unique(by[te])) < 2: continue
        clf = make_clf().fit(Xbag[tr], by[tr])
        aucs.append(roc_auc_score(by[te], clf.predict_proba(Xbag[te])[:, 1]))
    return dict(feature_set=feat_name, n_bags=len(bags),
                mil_meanpool_auc=np.mean(aucs) if aucs else np.nan,
                mil_meanpool_std=np.std(aucs) if aucs else np.nan)

mil_rows = [eval_mil_meanpool(df_all, n) for n in
            ["manual_all", "deep_morph", "deep_full"] if n in REGISTRY]
mil_df = pd.DataFrame(mil_rows)
mil_df.to_csv(os.path.join(OUTPUT_DIR, "mil_meanpool_baseline.csv"), index=False)
mil_df.round(3)

In [ ]:
# ---- TODO (later): attention-MIL ----------------------------------------------
# Replace mean-pool with a learned attention pooling so the model can weight the
# senescent sub-population within each bag. Sketch:
#
#   class AttentionMIL(nn.Module):
#       def __init__(self, d_in, d_hidden=128):
#           self.feat = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU())
#           self.attn = nn.Sequential(nn.Linear(d_hidden, 64), nn.Tanh(), nn.Linear(64, 1))
#           self.head = nn.Linear(d_hidden, 1)
#       def forward(self, bag):                  # bag: (n_cells, d_in)
#           h = self.feat(bag)
#           a = torch.softmax(self.attn(h), dim=0)   # (n_cells, 1) attention weights
#           z = (a * h).sum(0)                        # bag embedding
#           return self.head(z), a                    # logit + per-cell attention
#
# Train with donor-grouped folds; the attention weights `a` give an interpretable
# per-cell "senescence contribution" map for the thesis figures.
print("Attention-MIL left as a scaffold — mean-pool baseline above establishes the wiring.")

## 14. Thesis summary — aligning results with the narrative

A single consolidated table + the narrative claims it supports.

**How to read this for the thesis (Chapter 3):**

1. **Validation of deep morphology.** Compare `deep_morph` vs `manual_morph` (AUC and
   the ΔAUC contrast). If `deep_morph` matches or beats `manual_morph`, the O2-VAE
   mask embeddings are validated as morphological descriptors.
2. **Does intensity add?** `deep_morph+manual_intensity` vs `manual_all` and vs
   `deep_morph`. A positive ΔAUC argues intensity carries senescence signal beyond
   shape.
3. **End-to-end deep.** `deep_full` vs `manual_all`. The headline "deep beats manual"
   claim, if positive.
4. **Size vs shape.** The `deep_morph → deep_morph_area` ΔAUC and the area-only floor
   quantify how much is purely cell enlargement (a known confound from image rescaling).
5. **Categorical vs continuum.** Binary AUC (states) vs ordinal Spearman (continuum)
   + the PO check. Use the UMAP centroid trajectory as the visual companion.

Keep the binary single-cell results as the clean classic core; present ordinal and MIL
as extensions.

In [ ]:
summary = results_bin[["n_dims", "roc_auc", "pr_auc", "bal_acc"]].copy()
summary.columns = ["dims", "ROC_AUC", "PR_AUC", "bal_acc"]
print("=== Consolidated single-cell binary results ===")
display(summary.round(3))

print("\n=== Incremental ΔAUC contrasts ===")
display(delta_df.round(4))

print("\n=== Block permutation importance (best set) ===")
display(perm_df.round(4))

# one combined CSV for the thesis appendix
with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "thesis_results_summary.xlsx")) as xl:
    summary.to_excel(xl, sheet_name="binary")
    delta_df.to_excel(xl, sheet_name="delta_auc", index=False)
    perm_df.to_excel(xl, sheet_name="perm_importance")
    if "mil_df" in dir(): mil_df.to_excel(xl, sheet_name="mil_baseline", index=False)
print(f"\nSaved consolidated results to {OUTPUT_DIR}/thesis_results_summary.xlsx")